https://wimpouw.github.io/EnvisionBootcamp2021/MediaPipe_tophand_module

In [4]:
import os
import glob
import pandas as pd #data wrangling
import csv #csv saving

curfolder = os.getcwd()
projectdata = os.path.join(curfolder, "ToTrack")            # check to the folder that you need

vfiles = glob.glob(os.path.join(projectdata, "*", "*.mp4"), recursive=True) # list all mp4 files in the folder
vfiles = [f for f in vfiles if 'flipped' not in os.path.basename(f).lower()] # ignore if there is 'flipped' in the filename

#fleshfiles = glob.glob(os.path.join(projectdata, "*/*.avi"), recursive=True) 
outputf_mask = os.path.join(curfolder, "Output_Videos_hands")
if not os.path.exists(outputf_mask):
    os.makedirs(outputf_mask)
outtputf_ts = os.path.join(curfolder, "Output_TimeSeries")


#check videos to be processed
print("The following folder is set as the output folder where all the pose time series are stored")
print(os.path.abspath(outtputf_ts))
print("\n The following folder is set as the output folder for saving the masked videos ")
print(os.path.abspath(outputf_mask))
print("\n The following video(s) will be processed for masking: ")
print(vfiles)

The following folder is set as the output folder where all the pose time series are stored
e:\GtST_FLESH_KinTrajectory\Output_TimeSeries

 The following folder is set as the output folder for saving the masked videos 
e:\GtST_FLESH_KinTrajectory\Output_Videos_hands

 The following video(s) will be processed for masking: 
['e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_(AUF)PUMPEN.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_ABEND.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_ABHOLEN.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_ABLEHNEN.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_ABTROCKNEN .mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_ADLER.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_AFFE.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_AKKORDEON.mp4', 'e:\\GtST_FLESH_KinTrajectory\\ToTrack\\Gestures_trials\\GP02_AL

In [2]:
import mediapipe
import cv2


#initialize modules
drawingModule = mediapipe.solutions.drawing_utils #the module(s) usd from the mediapipe package
handsModule = mediapipe.solutions.hands           #the module(s) usd from the mediapipe package

In [3]:
################################some preperatory functions and lists for saving the data
#take some google classification object and convert it into a string
def makegoginto_str(gogobj):
    gogobj = str(gogobj).strip("[]")
    gogobj = gogobj.split("\n")
    return(gogobj[:-1]) #ignore last element as this has nothing

#Hand landmarks
markers = ['WRIST', 'THUMB_CMC', 'THUMB_MCP', 'THUMB_IP', 'THUMB_TIP', 
 'INDEX_MCP', 'INDEX_PIP', 'INDEX_DIP', 'INDEX_TIP', 
 'MIDDLE_MCP', 'MIDDLE_PIP', 'MIDDLE_DIP','MIDDLE_TIP', 
 'RING_MCP', 'RING_TIP', 'RING_DIP', 'RING_TIP', 
 'PINKY_MCP', 'PINKY_PIP', 'PINKY_DIP', 'PINKY_TIP']

#make the stringifyd position traces into clean values
def listpostions(newsamplemarks):
    tracking_p = []
    for value in newsamplemarks:
        stripped = value.split(':', 1)[1]
        stripped = stripped.strip() #remove spaces in the string if present
        tracking_p.append(stripped) #add to this list  
    return(tracking_p)

#a function that only retrieves the numerical info in a string
def only_numerics(seq):
    seq_type= type(seq)
    return seq_type().join(filter(seq_type.isdigit, seq))

In [5]:
#loop through the frames of the video

error_log = []

for ff in vfiles:

    print("We will now process video:")
    print(ff)
    print("This is video number " + str(vfiles.index(ff)) + " of " + str(len(vfiles)) + " videos in total")
    

    fname = ff.split(os.sep)[-1] #get the name of the file

    # outtputf_ts + fname + '_hands_world.csv' - if this exists, skip
    # if os.path.exists(outtputf_ts + fname + '_hands_world.csv'):
    #     print("This video has already been processed, skipping...")
    #     continue

    #capture the video and save some video properties
    capture = cv2.VideoCapture(ff)
    frameWidth = capture.get(cv2.CAP_PROP_FRAME_WIDTH)
    frameHeight = capture.get(cv2.CAP_PROP_FRAME_HEIGHT)
    fps = capture.get(cv2.CAP_PROP_FPS)

    #print(frameWidth, frameHeight, fps ) #print some video info to the console
    
    #make a video file where we will project keypoints on
    samplerate = fps #make the same as current 
    fourcc = cv2.VideoWriter_fourcc(*'XVID') #(*'XVID')
    out = cv2.VideoWriter(os.path.join(outputf_mask, fname[:-4]+'.avi'), fourcc, fps= samplerate, frameSize = (int(frameWidth), int(frameHeight))) #make sure that frameheight/width is the same a original

    #make a variable list with x, y, z, info where data is appended to
    markerxyz = []
    for mark in markers:
        for pos in ['X', 'Y', 'Z']:
            nm = pos + "_" + mark
            markerxyz.append(nm)
    addvariable = ['index', 'confidence', 'hand', 'time']
    addvariable.extend(markerxyz)
    time = 0
    fr = 1
    timeseries = [addvariable]
    timeseries_world = [addvariable] #for world coordinates
    #MAIN ROUTINE
         #For finetuning the tracking here check: https://google.github.io/mediapipe/solutions/hands.html
    with handsModule.Hands(static_image_mode=False,
                        model_complexity=1,
                        max_num_hands=2,
                        min_detection_confidence=0.5,
                        min_tracking_confidence=0.5) as hands:
        while (True):
            ret, frame = capture.read()
            if ret == True:
                results = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                # the results.multi_hand_landmarks should contain sets of x,y,z values for each landmark
                # However, they have no label or ID, just raw coordinates. 
                # we do know which set of coordinates corresponds to which joint:
                # see https://google.github.io/mediapipe/solutions/hands.html and figure 2.21 on that page

                ## NORMALIZED LANDMARKS IN PIXELS
                if results.multi_hand_landmarks != None: 
                    #attach an id based on location                    
                    for handLandmarks, handinfo in zip(results.multi_hand_landmarks,results.multi_handedness):
                        # these first few lines just convert the results output into something more workable
                        newsamplelmarks = makegoginto_str(handLandmarks.landmark)
                        newsamplelmarks = listpostions(newsamplelmarks)
                        newsampleinfo = makegoginto_str(handinfo) #get info the hands
                        # now we compile the data into a complete row, and add it to our dataframe
                        fuldataslice = [fr, newsampleinfo[2], newsampleinfo[3]]
                        fuldataslice.extend([str(time)]) #add time
                        fuldataslice.extend(newsamplelmarks) #add positions
                        timeseries.append(fuldataslice)
                        #get information about hand index [0], hand confidence [1], handedness [2] 
                        try:             
                            for point in handsModule.HandLandmark:
                                normalizedLandmark = handLandmarks.landmark[point]
                                # now draw the landmark onto the video frame
                                pixelCoordinatesLandmark = drawingModule._normalized_to_pixel_coordinates(normalizedLandmark.x, normalizedLandmark.y, frameWidth, frameHeight)
                                cv2.circle(frame, pixelCoordinatesLandmark, 5, (0, 255, 0), -1)
                        except Exception as e:
                            print("Error in processing landmarks for video: " + ff)
                            print(e)
                            error_log.append((ff, e))
                            continue
                if results.multi_hand_landmarks == None:
                    timeseries.append(["NA"]) #add a row of NAs

                ## NORMALIZE LANDMARKS IN WORLD COORDINATES
                if results.multi_hand_world_landmarks != None: 
                    #attach an id based on location                    
                    for handLandmarks, handinfo in zip(results.multi_hand_world_landmarks,results.multi_handedness):
                        # these first few lines just convert the results output into something more workable
                        newsamplelmarks = makegoginto_str(handLandmarks.landmark)
                        newsamplelmarks = listpostions(newsamplelmarks)
                        newsampleinfo = makegoginto_str(handinfo) #get info the hands
                        # now we compile the data into a complete row, and add it to our dataframe
                        fuldataslice = [fr, newsampleinfo[2], newsampleinfo[3]]
                        fuldataslice.extend([str(time)]) #add time
                        fuldataslice.extend(newsamplelmarks) #add positions
                        timeseries_world.append(fuldataslice)
                        #get information about hand index [0], hand confidence [1], handedness [2]              
                        #for point in handsModule.HandLandmark:
                         #   normalizedLandmark = handLandmarks.landmark[point]
                          #  # now draw the landmark onto the video frame
                           # pixelCoordinatesLandmark = drawingModule._normalized_to_pixel_coordinates(normalizedLandmark.x, normalizedLandmark.y, frameWidth, frameHeight)
                            #cv2.circle(frame, pixelCoordinatesLandmark, 5, (0, 255, 0), -1)
                if results.multi_hand_world_landmarks == None:
                    timeseries_world.append(["NA"]) #add a row of NAs

                    
                cv2.imshow('Test hand', frame)
                out.write(frame)  #########################################comment this out if you dont wn
                time = round(time+1000/samplerate)
                fr = fr+1
                if cv2.waitKey(1) == 27:
                    break
            if ret == False:
                break
    out.release()
    capture.release()
    cv2.destroyAllWindows()

    ####################################################### data to be written row-wise in csv file
    data = timeseries
    data_world = timeseries_world

    filehands = open(os.path.join(outtputf_ts, fname + '_handModel_pxl.csv'), 'w+', newline='')
    with filehands:
        write = csv.writer(filehands)
        write.writerows(data)

    # Save CSV data for world landmarks
    filehands_world = open(os.path.join(outtputf_ts, fname + '_handModel_world.csv'), 'w+', newline='')
    with filehands_world:
        write = csv.writer(filehands_world)
        write.writerows(data_world)

We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_(AUF)PUMPEN.mp4
This is video number 0 of 12398 videos in total
We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_ABEND.mp4
This is video number 1 of 12398 videos in total
We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_ABHOLEN.mp4
This is video number 2 of 12398 videos in total
We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_ABLEHNEN.mp4
This is video number 3 of 12398 videos in total
We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_ABTROCKNEN .mp4
This is video number 4 of 12398 videos in total
We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_ADLER.mp4
This is video number 5 of 12398 videos in total
We will now process video:
e:\GtST_FLESH_KinTrajectory\ToTrack\Gestures_trials\GP02_AFFE.mp4
This is video number 6 of 12398 videos